creates a live listener to telegram using Telethon. push new messeges into a kafka topic "raw".

In [ ]:
#setuop
!pip install telethon kafka-python-ng
#for this part you need to kafka running

Synchronizing state of docker.service with SysV service script with /usr/lib/systemd/systemd-sysv-install.
Executing: /usr/lib/systemd/systemd-sysv-install enable docker
[sudo] password for wnder: 
sudo: a password is required
^C


In [ ]:
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
import json
#create kafka topic

with open("config.json", "r") as f:
    config = json.load(f)

admin = KafkaAdminClient(bootstrap_servers=config["kafka"]["bootstrap_server"])

topics = [
    NewTopic(name=config["kafka"]["raw_topic"], num_partitions=config["kafka"]["raw_partitions"], replication_factor=1),
    NewTopic(name=config["kafka"]["tokens_topic"], num_partitions=config["kafka"]["tokens_partitions"], replication_factor=1)
]

for topic in topics:
    try:
        admin.create_topics([topic], validate_only=False)
        print(f"Created topic: '{topic.name}'")
    except TopicAlreadyExistsError:
        print(f"Topic '{topic.name}' already exists.")

admin.close()

ℹTopic 'telegram-raw' already exists


In [ ]:
import json
from telethon import TelegramClient, events
from kafka import KafkaProducer

API_ID = config["api_id"]
API_HASH = config["api_hash"]
CHANNELS = config["channels"]
TOPIC = config["kafka_topic"]["raw_topic"]


producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode('utf-8')
)

client = TelegramClient("tele_session", API_ID, API_HASH)

#what happens when a message arrives
@client.on(events.NewMessage(chats=CHANNELS))
async def handle_new_message(event):
    if not event.raw_text:
        return

    payload = {
        "text": event.raw_text,
        "ts": int(event.date.timestamp())
    }

    producer.send(TOPIC, value=payload)
    print(f"Sent to Kafka: {event.raw_text[:40]}...")

await client.start()
print("Listening to Telegram channels...")
await client.run_until_disconnected()

Server sent a very new message with ID 7677298478329968645, ignoring (see FAQ for details)
Server sent a very new message with ID 7677298478387210241, ignoring (see FAQ for details)
Server closed the connection: 0 bytes read on a total of 8 expected bytes
Server sent a very new message with ID 7677298873110358025, ignoring (see FAQ for details)
Server sent a very new message with ID 7677298873146374145, ignoring (see FAQ for details)
Server sent a very new message with ID 7677298873148473345, ignoring (see FAQ for details)
